# 第8周 · 周五：AB实验避坑大全——从"做实验"到"做对实验"

> 难度：⭐⭐⭐⭐ | 定位：系统掌握AB实验最常见的十大陷阱及其防御策略
> 前置依赖：W7-W8所有实验知识 | 后续用途：实验审核、实验平台治理、团队培训
> 面试地位：大厂面试高频题——"你做实验踩过哪些坑？"、"如何防止Peeking？"
> 配对课程：01python_learn W8D5（知识图谱与期中复习）

---

## 📘 区块1：核心概念与第一性原理

### 1.0 十大陷阱总览

AB实验的陷阱可以分为三类：**统计陷阱**（技术问题）、**心理陷阱**（人性问题）、**工程陷阱**（实现问题）。

| # | 坑 | 类型 | 后果 | 严重度 |
|---|-----|------|------|--------|
| 1 | Peeking | 统计+心理 | α从5%飙到20-30% | 🔴致命 |
| 2 | 多重检验 | 统计 | 20个指标→1个假阳性 | 🔴致命 |
| 3 | 指标偷换 | 心理 | 主指标不显著→换一个→假阳性 | 🔴致命 |
| 4 | SRM忽视 | 工程 | 分流bug→两组不可比→全盘作废 | 🔴致命 |
| 5 | 实验太短 | 统计 | 新奇效应→高估效果 | 🟡严重 |
| 6 | 只看p不看效应量 | 统计 | 大样本下0.01%也显著→无意义上线 | 🟡严重 |
| 7 | 忽略护栏指标 | 心理 | GMV涨了但利润率崩了 | 🟡严重 |
| 8 | 样本污染 | 工程 | 同一用户出现在两组→交叉污染 | 🟡严重 |
| 9 | 残留效应 | 统计 | 上个实验的影响还在→本实验有偏 | 🟢中等 |
| 10 | 新奇效应 | 统计 | 用户对新东西天然好奇→前期高估 | 🟢中等 |

### 1.1 为什么80%的陷阱是"人"的问题而不是"技术"的问题？

回顾这十个陷阱：Peeking（忍不住偷看）、多重检验（看了太多指标）、指标偷换（想看到好结果）、忽略护栏（只看到好的那一面）——这些都不是"统计学没学好"，而是"人性使然"。

每个人（包括分析师自己）都希望实验成功。这种希望会导致：
- 每天看一次p值（"说不定今天p就降下来了"）
- 看20个指标（"总有一个会显著吧"）
- 主指标不显著换一个（"这个指标也重要啊"）

**防御这些陷阱的第一道防线不是统计学知识，而是实验前就注册好：主要指标是什么、样本量多少、跑多久。这些"预注册"让你在实验中途无法"灵活调整"——因为调整了就破坏了统计保证。**

### 1.2 Peeking——最隐蔽的α膨胀

**为什么Peeking如此危险**：每次"偷看"p值，相当于做了一次假设检验。如果你打算跑2周但每天都看一次→相当于做了14次检验→即使H0为真（策略没效果），14次中至少有一次p<0.05的概率约为1-(1-0.05)^14≈51%！你的α从5%膨胀到了51%。

**Peeking的正确方式——序贯检验**：如果你确实需要中间分析（比如策略可能有害，需要提前停止），那就用序贯检验（Sequential Testing）。序贯检验在实验开始前就规划好中间分析的次数和时间点，并在每次分析时使用更严格的α阈值（如α_spending_function），保证总体的type I error控制在5%。

**简单记忆**：没规划过中间分析就看数据 = Peeking = α膨胀。规划过中间分析 + 用序贯检验 = 合法。

### 1.3 多重检验——"盯上"显著的那个指标

**为什么危险**：看了20个指标，每个α=0.05→期望有1个假阳性。然后你盯着那1个假阳性说"看！策略有效！"

**防御策略——预注册**：实验开始前，明确写出"主要指标是X，次要指标是Y和Z"。主要指标是决策依据，次要指标是参考信息。如果主要指标不显著，即使次要指标显著也不能上线——除非你把它注册为一个新实验的主要指标重新验证。

**Bonferroni校正的使用场景和代价**：
- 适用：你真的需要同时严格推断多个指标（如三个指标都需要做上线决策）
- 代价：α' = α/k（k=指标数），MDE会增大，样本量需求增加
- 不适用：探索性分析——次要指标"看起来显著"不应该触发Bonferroni，而应该触发一个新实验

### 1.4 指标偷换——最常被忽视的作弊

**为什么危险**：你的实验假设是"新算法能提升用户下单率"。结果下单率p=0.12（不显著），但点击率p=0.01（显著）。你说"那我们就说算法提升了点击率吧"。但问题是——点击率从来不是你做这个实验的原因。

**为什么这是作弊**：因为如果你当初要测的是点击率，样本量计算就会基于点击率的方差（而不是下单率的方差），样本量可能完全不同。你现在用下单率样本量做点击率检验——这个p值实际上是"后验挑选"的，α已经被膨胀了。

**防御策略——新实验原则**：如果你发现一个意料之外的显著效果，把它注册为一个新实验，用正确的指标重新计算样本量，重新跑。不要在同一次实验中"切换目标"。

### 1.5 SRM忽视——实验的"地基"不牢

SRM（Sample Ratio Mismatch，样本比例不匹配）意味着实际的两组人数比例显著偏离了设计的50:50。这可能意味着：
- 分流代码有bug（某些用户被错误排除）
- 机器人流量（机器人的ID和行为模式与真实用户不同）
- 实验组有更高的crash率（实验代码有bug导致部分用户被踢出）

**为什么每次实验都要查SRM**：SRM是实验有效性的前提。SRM异常的实验应该直接废弃——不需要看任何其他指标。

**检验方法**：卡方检验。H0：实际比例=设计比例（50:50）。如果p<0.01→SRM异常→调查原因。

### 1.6 实验太短——被"新奇效应"欺骗

用户对新产品/新功能有天然的好奇心。这个"好奇心"会在几天到几周内消退。如果实验只跑了3天，你测到的很可能是"用户好奇地试用了一下"的效果，而不是"用户会长期使用"的效果。

**如何确定实验长度**：
- 先看用户的行为周期：如果用户平均每周使用3次，一个完整的"使用周期"大约是1周
- 至少跑1-2个完整周期（即1-2周）
- 如果关心长期效果，至少跑4周
- 用"累积效果曲线"检查效果是否稳定——如果每天的效果都在衰减，可能还需要更长时间

### 1.7 只看p不看效应量——"显著但不重要"

大样本（如每组100万人）时，一个0.01%的提升也可能是显著的（p<0.05）。但0.01%的提升有业务意义吗？如果实验成本是10万元，0.01%的GMV提升每年多赚1万元——ROI是负的。

**正确做法**：同时看三个信息——p值（是否可能由随机性导致）、效应量（效果有多大）、置信区间（效果的可能范围）。只有当效应量 > MDE且 > 业务有意义提升时，"显著"才有业务价值。

### 1.8 忽略护栏指标——"拆东墙补西墙"

GMV涨了+5%——好消息！但客单价降了-8%？用户买了更多件但每件更便宜？利润率从25%降到18%？如果只看GMV不看护栏指标，你可能在用一个"发优惠券→GMV涨→利润跌→公司亏钱"的策略。

**每个实验必须配护栏指标**：
- 核心护栏：不能恶化的指标（如利润率、退款率、卸载率）
- 次要护栏：需要关注的指标（如客单价、用户满意度）
- 如果护栏指标显著恶化，即使主指标显著也不能上线

### 1.9 样本污染——"同一个用户在两组"

用户换了设备（手机→平板）、换了账号（新注册→老账号）、或者分流用了不稳定的ID（如cookie）→同一个真实用户可能被分到了两组。结果是两组的数据被"交叉污染"→组间差异被稀释→更难检测到真实效果。

**防御**：使用稳定的用户标识（user_id而非device_id或cookie），建立"用户-设备"映射表，定期清理"一用户多ID"的情况。

### 1.10 残留效应与新奇效应——时间维度的陷阱

**残留效应**：上个实验的影响还没消退就开始了新实验。比如上个实验给了实验组用户优惠券→他们还在用→新实验的"对照组"实际上还在受上个实验的影响。

**新奇效应**：用户对新东西有天然好奇→前期参与度高→但好奇心消退后回归正常→短期效果被高估。

**防御**：实验之间留"洗脱期"（washout period），至少跑1-2个完整用户周期让新奇效应消退。</cell>


## 🧩 区块2：知识点详细拆解——十大陷阱的深度剖析

### 2.1 Peeking的数学推导——α如何从5%膨胀到51%

假设每天看一次p值，只要p<0.05就停。在H0为真的情况下：
- 第1天停的概率 = 0.05（5%的人第一天就"显著"）
- 第1天不停、第2天停的概率 = 0.95 × 0.05 = 0.0475
- 前k天至少停一次的概率 = 1 - (1-0.05)^k

k=14天：1-(0.95)^14 ≈ 1-0.488 ≈ 51.2%！

这就是为什么Peeking如此危险——14天的Peeking让假阳性率从5%膨胀到51%。

### 2.2 多重检验的量化分析

实验通常看的不只是1个指标。如果看了20个指标（转化率、GMV、客单价、点击率、停留时长...），每个α=0.05：
- 预期假阳性数 = 20 × 0.05 = 1个
- 这1个假阳性看起来和其他真阳性一模一样——你无法区分

**FDR（False Discovery Rate）控制**：比Bonferroni更宽松的校正方法——不要求"没有假阳性"，只要求"假阳性比例控制在一定水平"。当你看20个指标时，FDR比Bonferroni更合适——因为你知道其中几个可能是真的。

### 2.3 指标偷换的统计后果——后验选择偏差

这是典型的"数据窥探"（Data Dredging）。统计上叫"Cherry Picking"或"HARKing"（Hypothesizing After the Results are Known——知道结果后再提假设）。

**问题**：你不是在"检验假设"，而是在"寻找故事"。任何数据集都有随机波动——20个指标中总有1个因为随机波动而看起来"显著"。你找到这个显著后编一个故事——这个故事在统计上是无效的。

### 2.4 SRM的统计检验与排查流程

卡方检验：χ² = Σ(O-E)²/E。自由度=组数-1。

SRM不通过时的排查步骤：
1. 检查实验配置——分流比例是否被错误修改？
2. 检查各组排除规则——是否某组被错误排除了部分用户？
3. 检查机器人流量——机器人是否被错误地更多分到某组？
4. 检查用户ID稳定性——是否某组用户的ID更容易丢失？

### 2.5 效应量与置信区间——p值的"配角"

Cohen's d = (均值差) / 合并标准差。是标准化的效应量：
- d=0.2：小效应（可能需要大样本才能检测）
- d=0.5：中等效应（大多数业务场景下有意义）
- d=0.8：大效应（明显的业务影响）

置信区间比p值包含更多信息——它不仅告诉你"是否显著"，还告诉你"效果可能在什么范围"。

### 2.6 护栏指标的设计原则

不是所有指标都适合作护栏：
- 必须对业务至关重要（如利润率、退款率、用户满意度）
- 必须能在实验周期内测量（长期指标不适合做护栏）
- 数量不宜太多（2-5个），否则又陷入多重检验问题

### 2.7 样本污染的量化和修复

通过检查"同一用户在两组"的比例来量化污染。如果超过2-3%，实验结果可能不可靠。修复方法：基于"首次出现"原则——用户在哪个组第一次出现，就把所有后续行为归到那个组。

### 2.8 实验清单——每次审查实验的10个问题

1. SRM检查通过了吗？
2. 主要指标是什么？（是否预注册？）
3. Peeking被控制了吗？（是否做了序贯检验规划？）
4. 多重检验被校正了吗？
5. 护栏指标被监控了吗？
6. 样本量达到MDE要求了吗？
7. 实验跑够时间了吗？（用户周期×1-2？）
8. 效应量有业务意义吗？
9. 有样本污染的迹象吗？
10. 上次实验的残留效应被排除了吗？</cell>


## 💻 区块3：动手练习——识别和量化实验陷阱

In [ ]:
## 📝 区块7：当日小结

> **今天最重要的一个认知**：AB实验的陷阱，80%不是技术问题，是"人想看到好结果"的心理问题。Peeking→等p降下来，指标偷换→换到显著的那个，忽略护栏→只看好的那一面。防御这些陷阱的第一道防线不是统计学知识，而是实验前预注册（主要指标、样本量、跑多久）和实验后检查清单（SRM→主指标→效应量→护栏→Peeking）。

**五条金句带回家**：
1. Peeking让α从5%膨胀到51%——14天每天看p值的实验，一半以上是假阳性
2. 看20个指标时，至少一个"显著"的概率是64%——不是策略好，是随机在玩你
3. 主指标不显著换一个指标 = 作弊——把它注册为新实验重新验证
4. 护栏指标是"刹车"——没有护栏，你只看得到加速，看不到撞墙
5. 诚实比聪明更难——不做Peeking/指标偷换/忽略护栏，就是最大的职业素养

---

*03analyst_thinking · Week 8 Day 5 · AB实验避坑大全 · 2026-05-21*</cell>


## 📊 区块4：业务场景应用

### 场景1：一个实验被"Peeking"毁掉的真实案例

某电商公司测试"会员优惠券"策略，设计跑2周。运营总监每天早上一到公司就打开看板看p值。第6天p=0.04→"显著！停实验，全量上线！"

全量上线后3个月，数据团队做回顾分析：真实效果只有+0.3%，远低于最初测到的+5%。最初的"显著"是Peeking+小样本下的随机波动制造的幻象。

**教训**：这个案例中，错误的决策不是"上线了一个没用的大策略"，而是"把一件本来应该跑2周才能下结论的事，在第6天就仓促决定了"。

### 场景2：护栏指标救了公司

某游戏公司测试"降低首充门槛"策略。主指标（首充率）大幅提升+30%，p<0.001——所有人都很兴奋，准备全量。

但分析师注意到护栏指标：7日留存从45%降到38%（p<0.001），30日LTV从120元降到85元（p<0.001）。深挖发现：低门槛吸引来了大量"充6元就不玩了"的用户——首充率涨了但用户质量暴跌。

**决策**：不上线。虽然主指标"好看"，但护栏指标暴露了策略的长期伤害。

**教训**：护栏指标是实验决策的"刹车"——没有护栏，你只看得到"加速"，看不到"撞墙"。

### 场景3：SRM发现分流bug

某平台的一个实验：预期50:50分流，实际分了53:47。SRM p<0.0001→严重异常。

排查发现：实验组的代码有一个bug，导致iOS用户在特定场景下crash→这些用户被"踢出"了实验→实验组的iOS用户变少了。因为iOS用户的消费习惯和Android用户不同→两组不可比→实验结果全盘作废。

**教训**：SRM检查只需5秒（看卡方检验p值），但不检查的代价是"浪费了2周的实验时间和几十万用户的数据"。</cell>


## ✅ 区块5：分级检验清单

### 🟢 初级检验
- [ ] 能说出10个常见陷阱的名字（至少8个）
- [ ] 能解释Peeking为什么导致α膨胀
- [ ] 能解释为什么看20个指标会有假阳性问题
- [ ] 能在每次看实验结果前检查SRM

### 🟡 中级检验
- [ ] 能用量化例子说明Peeking的α膨胀量级（如14天从5%到51%）
- [ ] 能区分预注册指标、次要指标、探索性指标
- [ ] 能说出至少3个护栏指标的例子
- [ ] 能写出SRM的卡方检验

### 🔴 高级检验
- [ ] 能审查一个实验报告并识别所有陷阱
- [ ] 能向产品经理解释"为什么不能主指标不显著换一个指标"
- [ ] 能设计实验的Peeking防御策略（序贯检验 or 固定样本量）
- [ ] 能建立团队的"实验审查清单"</cell>


## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W7假设检验 | Peeking/多重检验是假设检验的误用 |
| **← 前置依赖** | W8D1 MDE | 效应量要和MDE对比才有意义 |
| **← 前置依赖** | W8D2 CUPED | CUPED减少方差但不防Peeking |
| **→ 后续用途** | 实验审核流程 | 审查清单来自这10个陷阱 |
| **→ 后续用途** | 团队培训 | 新人入职第一课：先学陷阱再学方法 |
| **业务连接** | 所有实验决策 | 每个实验都需要经过这10个检查点 |

### 横向连接：陷阱之间的交互

最危险的组合：
- Peeking + 多重检验 + 指标偷换 = "三连击"
- 每天偷看→盯上显著的那个→如果主指标不显著就换一个→假阳性概率接近100%</cell>


---

## 四、当日小结

> **今天最重要的一个认知**：AB实验的坑，80%不是技术问题，是"人想看到好结果"的心理问题。Peeking→等p降下来，指标偷换→换到显著的那个。诚实比聪明更难。

---

*03analyst_thinking · Week 8 Day 5 · 2026-05-21*